De identification of DICOM files

In [1]:
!pip install pydicom

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install pypdf

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install pymupdf

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import fitz
import re
from pypdf import PdfReader

In [5]:
import os

# ── Local paths (point to actual data on disk) ──────────────────────────────
BASE_DIR     = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))  # project root
DICOM_INPUT  = os.path.join(BASE_DIR, "pseudo_phi_dicom_data")
DICOM_OUTPUT = os.path.join(BASE_DIR, "notebooks/data", "output", "DICOMs")
PDF_OUTPUT   = os.path.join(BASE_DIR, "notebooks/data", "output", "PDFs")
PDF_INPUT    = os.path.join(BASE_DIR, "synthetic-phi-generator/synthetic-phi-generator/data/input/pdf")

os.makedirs(DICOM_OUTPUT, exist_ok=True)
os.makedirs(PDF_OUTPUT, exist_ok=True)

print("DICOM_INPUT:", DICOM_INPUT)
print("Exists:", os.path.exists(DICOM_INPUT))
print("Folders configured successfully")


DICOM_INPUT: c:\Users\Yuvika Agrawal\OneDrive\Desktop\Workings\opencv\healthcare-deidentification-pipeline\healthcare-deidentification-pipeline\pseudo_phi_dicom_data
Exists: True
Folders configured successfully


In [6]:

dicom_files = []

for file in os.listdir(DICOM_INPUT):
    path = os.path.join(DICOM_INPUT, file)

    if os.path.isfile(path):
        dicom_files.append(path)

print("Total DICOM files:", len(dicom_files))

Total DICOM files: 1678


In [7]:
pdf_files = []

for file in os.listdir(PDF_INPUT):
    path = os.path.join(PDF_INPUT, file)

    if os.path.isfile(path):
        pdf_files.append(path)

print("Total PDF files:", len(pdf_files))

Total PDF files: 60


In [8]:
print("First 10 DICOM files:")
for file in dicom_files[:10]:
    print(os.path.basename(file))

print("\nFirst 10 PDF files:")
for file in pdf_files[:10]:
    print(os.path.basename(file))

First 10 DICOM files:
0002bbff-d80b-4694-a247-a305056ce52e.dcm
0012c394-5530-4879-9560-d173970f5acb.dcm
001386d0-7645-46a8-8c52-3256e91a15ea.dcm
002797a0-bca7-40c6-b7c9-182692c8f5f3.dcm
003f9682-64bf-49c1-a7a8-c352d3cd8471.dcm
003ffd00-eb4c-4a54-981f-063e2699a932.dcm
0042f584-74a9-44eb-a400-3902cba9e629.dcm
0043af2a-63c2-4124-9701-35d642426fb2.dcm
004eb7fe-d2ba-4a99-83f5-4f14026bdfa0.dcm
006cc595-d644-40b3-80fb-bf4b13c005a6.dcm

First 10 PDF files:
lab_03213.pdf
lab_13676.pdf
lab_35683.pdf
lab_37204.pdf
lab_39281.pdf
lab_41398.pdf
lab_45264.pdf
lab_52563.pdf
lab_59709.pdf
lab_69748.pdf


Working on one file first

In [9]:
import pydicom

sample_file = dicom_files[0]

dicom = pydicom.dcmread(sample_file)

print("DICOM loaded successfully")


DICOM loaded successfully


In [10]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import hmac
import hashlib
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from project root

SALT = os.environ.get("DEID_SALT", "local-dev-salt-do-not-use-in-prod").encode("utf-8")

def hmac_hex(value: str) -> str:
    return hmac.new(SALT, value.encode("utf-8"), hashlib.sha256).hexdigest()

print("SALT loaded successfully")


SALT loaded successfully


In [12]:
original_patient_id = dicom.PatientID
pseudo_patient_id = "ANON" + hmac_hex(str(original_patient_id))[:16].upper()

dicom.PatientID = pseudo_patient_id

print("Original:", original_patient_id)
print("Pseudonymous:", pseudo_patient_id)

Original: 8834647487
Pseudonymous: ANONFE74D56C38C6C345


In [13]:
#check
d1 = pydicom.dcmread(dicom_files[0])
d2 = pydicom.dcmread(dicom_files[1])
print(hmac_hex(str(d1.PatientID)) == hmac_hex(str(d2.PatientID)))

False


In [14]:
dicom.PatientName = "ANONYMOUS"
print(dicom.PatientName)

ANONYMOUS


In [15]:
def get_date_offset(patient_id: str) -> int:
    digest = hmac_hex(str(patient_id))
    raw_int = int(digest[:8], 16)   # first 8 hex chars from a patient_id, convert to a number for every date field belonging to this patient.
    offset = (raw_int % 729) - 364  # squash into range (-364,364)
    return offset

offset_days = get_date_offset(original_patient_id)
print("This patient's date offset:", offset_days, "days")

This patient's date offset: 114 days


In [16]:
from datetime import datetime, timedelta

if "PatientBirthDate" in dicom:
    raw_date = str(dicom.PatientBirthDate)   #"YYYYMMDD"

    if raw_date:
        parsed_date = datetime.strptime(raw_date, "%Y%m%d")
        shifted_date = parsed_date + timedelta(days=offset_days)
        dicom.PatientBirthDate = shifted_date.strftime("%Y%m%d")

print("Shifted birth date:", dicom.get("PatientBirthDate"))

Shifted birth date: 19660103


In [17]:
# not needed fields for analysis
remove_fields = [
    "PatientAddress",
    "PatientTelephoneNumbers",
    "ReferringPhysicianName",
    "PerformingPhysicianName",
    "InstitutionName",
    "InstitutionAddress",
    "OperatorsName",
    "StationName",
    "OtherPatientIDs",
    "OtherPatientNames",
    "AccessionNumber",
    "StudyID",
]

removed_count = 0
for field in remove_fields:
    if field in dicom:
        del dicom[field]
        removed_count += 1

print(f"Removed {removed_count} of {len(remove_fields)} fields (rest weren't present in this file)")

Removed 6 of 12 fields (rest weren't present in this file)


In [18]:
#sanity check
for field in remove_fields:
    print(field, ":", "still present" if field in dicom else "removed or absent")

PatientAddress : removed or absent
PatientTelephoneNumbers : removed or absent
ReferringPhysicianName : removed or absent
PerformingPhysicianName : removed or absent
InstitutionName : removed or absent
InstitutionAddress : removed or absent
OperatorsName : removed or absent
StationName : removed or absent
OtherPatientIDs : removed or absent
OtherPatientNames : removed or absent
AccessionNumber : removed or absent
StudyID : removed or absent


In [19]:
before_count = len(dicom)
dicom.remove_private_tags()
after_count = len(dicom)

print(f"Tag count before: {before_count}")
print(f"Tag count after: {after_count}")
print(f"Private tags removed: {before_count - after_count}")

Tag count before: 112
Tag count after: 105
Private tags removed: 7


In [20]:
def check_burned_in_risk(dicom):
    modality = str(dicom.get("Modality", ""))
    burned_in_flag = str(dicom.get("BurnedInAnnotation", "")).upper()

    risky_modalities = ["US", "OT", "SC"]  # ultrasound, other, secondary capture

    if burned_in_flag == "YES":
        return True, "BurnedInAnnotation tag explicitly set to YES"
    if modality in risky_modalities:
        return True, f"modality '{modality}' has known burned-in-PHI risk, flag unset/unreliable"

    return False, None

Final Function including all sanity checks

In [21]:
def deidentify_dicom(dicom):
    original_patient_id = dicom.PatientID if "PatientID" in dicom else "UNKNOWN"

    hashed_count = 0
    date_shifted_count = 0
    removed_count = 0

    # pseudonymous PatientID
    pseudo_patient_id = "ANON" + hmac_hex(str(original_patient_id))[:16].upper()
    dicom.PatientID = pseudo_patient_id
    hashed_count += 1

    # hash UIDs
    uid_fields = ["StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID", "FrameOfReferenceUID"]
    for uid_field in uid_fields:
        if uid_field in dicom:
            original_uid = str(dicom.get(uid_field))
            digest = hmac_hex(original_uid)
            new_uid = f"2.25.{int(digest, 16)}"[:64]
            setattr(dicom, uid_field, new_uid)
            hashed_count += 1

    # PatientName
    dicom.PatientName = "ANONYMOUS"

    # shift dates
    offset_days = get_date_offset(original_patient_id)
    date_fields = ["PatientBirthDate", "StudyDate", "SeriesDate"]
    for date_field in date_fields:
        if date_field in dicom:
            raw_date = str(dicom.get(date_field))
            if raw_date:
                try:
                    parsed_date = datetime.strptime(raw_date, "%Y%m%d")
                    shifted_date = parsed_date + timedelta(days=offset_days)
                    setattr(dicom, date_field, shifted_date.strftime("%Y%m%d"))
                    date_shifted_count += 1
                except ValueError:
                    setattr(dicom, date_field, "")

    # remove direct-identifier
    def scrub_dataset(ds):
        nonlocal removed_count
        for field in remove_fields:
            if field in ds:
                del ds[field]
                removed_count += 1
        for elem in ds:
            if elem.VR == "SQ":
                for item in elem.value:
                    scrub_dataset(item)

    scrub_dataset(dicom)

    # strip private tags
    dicom.remove_private_tags()

    return pseudo_patient_id, removed_count, hashed_count, date_shifted_count

In [22]:
import hashlib

def get_file_hash(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

def make_output_filename(pseudo_id: str, original_path: str) -> str:
    path_hash = hashlib.sha256(original_path.encode("utf-8")).hexdigest()[:10]
    ext = os.path.splitext(original_path)[1]
    return f"{pseudo_id}_{path_hash}{ext}"

Setting up the metadata database

In [23]:
import sqlite3

DB_PATH = "deid_pipeline.db"   # a plain file in your project folder - inspect anytime with any SQLite viewer

conn = sqlite3.connect(DB_PATH, isolation_level=None)  # isolation_level=None = autocommit, matches original behavior
conn.execute("PRAGMA foreign_keys = ON")  # SQLite has FK constraints but they're OFF by default - must enable explicitly
cur = conn.cursor()

cur.execute("SELECT 1")
print("Connected:", cur.fetchone() == (1,))

Connected: True


In [24]:
cur.execute("""
CREATE TABLE IF NOT EXISTS patients (
    pseudo_patient_id   TEXT PRIMARY KEY,
    sex                 TEXT,
    age                 TEXT,
    first_seen_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    last_seen_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS dicom_files (
    file_id                 INTEGER PRIMARY KEY AUTOINCREMENT,
    pseudo_patient_id       TEXT REFERENCES patients(pseudo_patient_id),
    source_file_hash        TEXT UNIQUE NOT NULL,
    output_filename          TEXT NOT NULL,
    modality                TEXT,
    manufacturer             TEXT,
    manufacturer_model       TEXT,
    body_part_examined       TEXT,
    rows_px                  INTEGER,
    columns_px                INTEGER,
    pixel_spacing            TEXT,
    tags_removed             INTEGER,
    tags_hashed               INTEGER,
    tags_date_shifted         INTEGER,
    quarantined               BOOLEAN DEFAULT 0,
    processed_at              TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS pdf_reports (
    file_id                 INTEGER PRIMARY KEY AUTOINCREMENT,
    pseudo_patient_id       TEXT REFERENCES patients(pseudo_patient_id),
    source_file_hash        TEXT UNIQUE NOT NULL,
    output_filename          TEXT NOT NULL,
    redaction_verified       BOOLEAN,
    processed_at              TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS pipeline_runs (
    run_id               INTEGER PRIMARY KEY AUTOINCREMENT,
    run_type             TEXT NOT NULL,
    started_at            TIMESTAMP,
    completed_at           TIMESTAMP,
    files_processed        INTEGER DEFAULT 0,
    files_skipped           INTEGER DEFAULT 0,
    files_quarantined       INTEGER DEFAULT 0,
    files_failed             INTEGER DEFAULT 0
);
""")

print("Tables created")

Tables created


In [25]:
def upsert_patient(pseudo_patient_id: str, sex: str = None, age: str = None):
    cur.execute("""
        INSERT INTO patients (pseudo_patient_id, sex, age, first_seen_at, last_seen_at)
        VALUES (?, ?, ?, CURRENT_TIMESTAMP, CURRENT_TIMESTAMP)
        ON CONFLICT (pseudo_patient_id) DO UPDATE
        SET last_seen_at = CURRENT_TIMESTAMP;
    """, (pseudo_patient_id, sex, age))


def insert_dicom_record(record: dict):
    cur.execute("""
        INSERT INTO dicom_files (
            pseudo_patient_id, source_file_hash, output_filename,
            modality, manufacturer, manufacturer_model, body_part_examined,
            rows_px, columns_px, pixel_spacing,
            tags_removed, tags_hashed, tags_date_shifted, quarantined
        ) VALUES (
            :pseudo_patient_id, :source_file_hash, :output_filename,
            :modality, :manufacturer, :manufacturer_model, :body_part_examined,
            :rows_px, :columns_px, :pixel_spacing,
            :tags_removed, :tags_hashed, :tags_date_shifted, :quarantined
        )
        ON CONFLICT (source_file_hash) DO NOTHING;
    """, record)


def insert_pdf_record(record: dict):
    cur.execute("""
        INSERT INTO pdf_reports (
            pseudo_patient_id, source_file_hash, output_filename, redaction_verified
        ) VALUES (
            :pseudo_patient_id, :source_file_hash, :output_filename, :redaction_verified
        )
        ON CONFLICT (source_file_hash) DO NOTHING;
    """, record)

Processing and saving all de-identified dicom files

In [26]:
import shutil
from datetime import datetime

QUARANTINE_DIR = "data/quarantine/DICOMs"
os.makedirs(QUARANTINE_DIR, exist_ok=True)

def is_already_processed_dicom(file_hash: str) -> bool:
    cur.execute("SELECT 1 FROM dicom_files WHERE source_file_hash = ?", (file_hash,))
    return cur.fetchone() is not None

summary = {"processed": 0, "skipped": 0, "quarantined": 0, "failed": 0}
failed_files = []
run_started = datetime.now()

for path in dicom_files:
    try:
        file_hash = get_file_hash(path)

        if is_already_processed_dicom(file_hash):
            summary["skipped"] += 1
            continue

        dicom = pydicom.dcmread(path)

        is_risky, reason = check_burned_in_risk(dicom)
        if is_risky:
            shutil.copy2(path, os.path.join(QUARANTINE_DIR, os.path.basename(path)))
            summary["quarantined"] += 1
            continue

        pseudo_id, removed_count, hashed_count, date_shifted_count = deidentify_dicom(dicom)

        out_filename = make_output_filename(pseudo_id, path)
        out_path = os.path.join(DICOM_OUTPUT, out_filename)
        dicom.save_as(out_path)

        upsert_patient(pseudo_id, sex=str(dicom.get("PatientSex", "")), age=str(dicom.get("PatientAge", "")))

        insert_dicom_record({
            "pseudo_patient_id": pseudo_id,
            "source_file_hash": file_hash,
            "output_filename": out_filename,
            "modality": str(dicom.get("Modality", "")),
            "manufacturer": str(dicom.get("Manufacturer", "")),
            "manufacturer_model": str(dicom.get("ManufacturerModelName", "")),
            "body_part_examined": str(dicom.get("BodyPartExamined", "")),
            "rows_px": int(dicom.get("Rows", 0)) or None,
            "columns_px": int(dicom.get("Columns", 0)) or None,
            "pixel_spacing": str(dicom.get("PixelSpacing", "")),
            "tags_removed": removed_count,
            "tags_hashed": hashed_count,
            "tags_date_shifted": date_shifted_count,
            "quarantined": False,
        })

        summary["processed"] += 1
    except Exception as e:
        summary["failed"] += 1
        failed_files.append((os.path.basename(path), str(e)))

cur.execute("""
    INSERT INTO pipeline_runs (run_type, started_at, completed_at, files_processed, files_skipped, files_quarantined, files_failed)
    VALUES ('dicom', ?, CURRENT_TIMESTAMP, ?, ?, ?, ?)
""", (run_started.isoformat(), summary["processed"], summary["skipped"], summary["quarantined"], summary["failed"]))

print("=== DICOM batch complete ===")
print(summary)
print(f"Total accounted for: {sum(summary.values())} / {len(dicom_files)}")
if failed_files:
    print("\nFailed files:")
    for name, error in failed_files:
        print(f"  {name}: {error}")

=== DICOM batch complete ===
{'processed': 1678, 'skipped': 0, 'quarantined': 0, 'failed': 0}
Total accounted for: 1678 / 1678


Checking the saved dicom outputs

In [28]:

output_files = os.listdir(DICOM_OUTPUT)

import random
sample_outputs = random.sample(output_files,7 )
for fname in sample_outputs:
    d = pydicom.dcmread(os.path.join(DICOM_OUTPUT, fname))
    print(f"\n{fname}")
    print(f"  PatientName: {d.get('PatientName')}")
    print(f"  PatientID: {d.get('PatientID')}")
    print(f"  PatientBirthDate: {d.get('PatientBirthDate', 'not present')}")
    print(f"  Modality: {d.get('Modality')}, Manufacturer: {d.get('Manufacturer')}")

# Check for Duplicate Patient IDS
from collections import Counter
pseudo_ids = [f.split("_")[0] for f in output_files]
id_counts = Counter(pseudo_ids)
repeats = {pid: count for pid, count in id_counts.items() if count > 1}

# Result
print(f"\n{len(repeats)} pseudonymous patient IDs appear in more than one file "
      f"(i.e. {sum(repeats.values())} files belong to {len(repeats)} repeat patients)")


ANON7E2BBF4A605025F3_d326de3bbe.dcm
  PatientName: ANONYMOUS
  PatientID: ANON7E2BBF4A605025F3
  PatientBirthDate: 18840628
  Modality: PT, Manufacturer: GE MEDICAL SYSTEMS

ANON8151EEC94CC5DAA5_2f1932f28f.dcm
  PatientName: ANONYMOUS
  PatientID: ANON8151EEC94CC5DAA5
  PatientBirthDate: 19380622
  Modality: PT, Manufacturer: GE MEDICAL SYSTEMS

ANONFE74D56C38C6C345_cf5273b7ae.dcm
  PatientName: ANONYMOUS
  PatientID: ANONFE74D56C38C6C345
  PatientBirthDate: 19660103
  Modality: PT, Manufacturer: GE MEDICAL SYSTEMS

ANON907019FDFA8CE554_786d01edd9.dcm
  PatientName: ANONYMOUS
  PatientID: ANON907019FDFA8CE554
  PatientBirthDate: 19310816
  Modality: MR, Manufacturer: SIEMENS

ANON4F05FD07A7DF3C76_c0b09c7d63.dcm
  PatientName: ANONYMOUS
  PatientID: ANON4F05FD07A7DF3C76
  PatientBirthDate: 19041009
  Modality: PT, Manufacturer: SIEMENS

ANON7E2BBF4A605025F3_4bb0cf9178.dcm
  PatientName: ANONYMOUS
  PatientID: ANON7E2BBF4A605025F3
  PatientBirthDate: 18840628
  Modality: PT, Manufacture

De-Identification of PDFs

In [29]:
sample_pdf = pdf_files[0]

reader = PdfReader(sample_pdf)
print(f"File: {os.path.basename(sample_pdf)}")
print(f"Pages: {len(reader.pages)}")
print("\n--- Extracted text from page 1 ---\n")
print(reader.pages[0].extract_text())

File: lab_03213.pdf
Pages: 1

--- Extracted text from page 1 ---

+
Sunridge Community Hospital
Patient ID : 03213
Patient Name : Sana Wallace
Gender : Female
Patient Age : 87 years
Collection Date : 19 Jul 2025
Ordering Physician : Dr. Noah Dubois
Contact : (208) 592-0893
Complete Blood Count
Test
Result
Units
Reference Range
Hemoglobin
13.8
g/dL
13.0 - 17.0
WBC Count
14.6
x10^3/uL
4.0 - 11.0
Platelet Count
250
x10^3/uL
150 - 400
Hematocrit
41
%
38 - 50
Interpretation
Mild leukocytosis noted; clinical correlation for possible infection recommended.



In [30]:
doc = fitz.open(sample_pdf)
page = doc[0]

# Sorting the extracted texts
text_sorted = page.get_text("text", sort=True)
print(text_sorted)

           Sunridge Community Hospital          +


Patient ID : 03213
Patient Name : Sana Wallace
Gender : Female
Patient Age : 87 years
Collection Date : 19 Jul 2025
Ordering Physician : Dr. Noah Dubois
Contact : (208) 592-0893

Complete Blood Count

Test                           Result              Units              Reference Range


Hemoglobin                      13.8                 g/dL                 13.0 - 17.0
WBC Count                      14.6                x10^3/uL             4.0 - 11.0
Platelet Count                 250                 x10^3/uL           150 - 400
Hematocrit                    41          %                 38 - 50

Interpretation
Mild leukocytosis noted; clinical correlation for possible infection recommended.


In [31]:
import re

KNOWN_LABELS = [
    "Patient ID", "Patient Age", "Patient Name", "GA", "Gender", "BMI",
    "Examination Findings", "Head", "Brain", "Heart", "Spine",
    "Abdominal wall", "Urinary tract", "Extremities", "Conclusion",
    "Collection Date", "Ordering Physician", "Contact",
]
label_alternation = "|".join(re.escape(label) for label in KNOWN_LABELS)

def extract_field(text: str, label: str) -> str:
    
    pattern = rf"{re.escape(label)}\s*:\s*([^\n]+?)(?=\s+(?:{label_alternation})\s*:|\n|\Z)"
    match = re.search(pattern, text)
    return match.group(1).strip() if match else None


def extract_header_institution(text: str) -> str:
    """The hospital/institution name is printed as page header text, BEFORE
    the first known label - it's not itself in 'Label : value' form, so it
    needs a different extraction approach than extract_field(). We take
    everything before the first known label appears, then strip out
    non-letter characters (logo glyphs sometimes extract as stray symbols
    like '+' next to the hospital name)."""
    first_label_pos = len(text)
    for label in KNOWN_LABELS:
        idx = text.find(label)
        if idx != -1:
            first_label_pos = min(first_label_pos, idx)
    header_text = text[:first_label_pos]
    cleaned = re.sub(r"[^A-Za-z\s]", " ", header_text).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned if cleaned else None


patient_id = extract_field(text_sorted, "Patient ID")
patient_name = extract_field(text_sorted, "Patient Name")
institution_name = extract_header_institution(text_sorted)

print("Extracted Patient ID:", patient_id)
print("Extracted Patient Name:", patient_name)
print("Extracted Institution:", institution_name)


Extracted Patient ID: 03213
Extracted Patient Name: Sana Wallace
Extracted Institution: Sunridge Community Hospital


Extracts, redacts, and save PDF. Returns a metadata or None if this exact file content was already processed.

In [32]:
PDF_QUARANTINE_DIR = "data/quarantine/pdf"
os.makedirs(PDF_QUARANTINE_DIR, exist_ok=True)

def deidentify_pdf(path: str):
    file_hash = get_file_hash(path)

    cur.execute("SELECT 1 FROM pdf_reports WHERE source_file_hash = ?", (file_hash,))
    if cur.fetchone() is not None:
        return {"status": "skipped"}

    doc = fitz.open(path)
    text_sorted = doc[0].get_text("text", sort=True)

    patient_id = extract_field(text_sorted, "Patient ID")
    patient_name = extract_field(text_sorted, "Patient Name")

    if not patient_id or not patient_name:
        doc.close()
        shutil.copy2(path, os.path.join(PDF_QUARANTINE_DIR, os.path.basename(path)))
        return {"status": "quarantined", "reason": "field_extraction_failed"}

    
    physician_name = extract_field(text_sorted, "Ordering Physician")
    contact = extract_field(text_sorted, "Contact")
    collection_date = extract_field(text_sorted, "Collection Date")
    institution_name = extract_header_institution(text_sorted)

    pseudo_patient_id = "ANON" + hmac_hex(str(patient_id))[:16].upper()

    values_to_redact = [v for v in
        [patient_id, patient_name, physician_name, contact, collection_date, institution_name]
        if v]

    for page in doc:
        for value in values_to_redact:
            for rect in page.search_for(value):
                page.add_redact_annot(rect, fill=(0, 0, 0))
        page.apply_redactions()

    out_filename = make_output_filename(pseudo_patient_id, path)
    out_path = os.path.join(PDF_OUTPUT, out_filename)
    doc.save(out_path)
    doc.close()

    verify_doc = fitz.open(out_path)
    verify_text = " ".join(page.get_text("text", sort=True) for page in verify_doc)
    verify_doc.close()
    # verify ALL redacted values are gone, not just patient_id/name
    redaction_verified = all(value not in verify_text for value in values_to_redact)

    upsert_patient(pseudo_patient_id)

    insert_pdf_record({
        "pseudo_patient_id": pseudo_patient_id,
        "source_file_hash": file_hash,
        "output_filename": out_filename,
        "redaction_verified": redaction_verified,
    })

    return {"status": "processed", "redaction_verified": redaction_verified}


In [33]:
summary = {"processed": 0, "skipped": 0, "quarantined": 0, "failed": 0}
verification_warnings = []
failed_files = []
run_started = datetime.now()

for path in pdf_files:
    try:
        result = deidentify_pdf(path)
        summary[result["status"]] += 1
        if result["status"] == "processed" and not result["redaction_verified"]:
            verification_warnings.append(os.path.basename(path))
    except Exception as e:
        summary["failed"] += 1
        failed_files.append((os.path.basename(path), str(e)))

cur.execute("""
    INSERT INTO pipeline_runs (run_type, started_at, completed_at, files_processed, files_skipped, files_quarantined, files_failed)
    VALUES ('pdf', ?, CURRENT_TIMESTAMP, ?, ?, ?, ?)
""", (run_started.isoformat(), summary["processed"], summary["skipped"], summary["quarantined"], summary["failed"]))

print("=== PDF batch complete ===")
print(summary)
print(f"Total accounted for: {sum(summary.values())} / {len(pdf_files)}")
if verification_warnings:
    print(f"\n{len(verification_warnings)} processed but redaction not verified: {verification_warnings}")
if failed_files:
    print("\nFailed files:")
    for name, err in failed_files:
        print(f"  {name}: {err}")

=== PDF batch complete ===
{'processed': 60, 'skipped': 0, 'quarantined': 0, 'failed': 0}
Total accounted for: 60 / 60
